In [3]:
!pip install yfinance pandas numpy requests fredapi python-dotenv matplotlib

In [4]:
import yfinance as yf

df = yf.download("JPM", period="5d")
print(df)

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker             JPM         JPM         JPM         JPM       JPM
Date                                                                
2026-06-15  319.399994  325.920013  318.839996  323.920013   7988900
2026-06-16  331.140015  331.750000  324.019989  324.299988  11094900
2026-06-17  333.459991  337.769989  331.500000  332.179993  12246400
2026-06-18  325.220001  338.089996  324.160004  336.950012  20111000
2026-06-22  331.480011  332.769989  326.815002  329.700012  10150827


In [5]:
from fredapi import Fred

fred = Fred(api_key="793ea442d5187f9e5b7b0c30de96367a")
data = fred.get_series("DGS3MO")
print(data.tail())

2026-06-12    3.78
2026-06-15    3.79
2026-06-16    3.79
2026-06-17    3.83
2026-06-18    3.83
dtype: float64


In [6]:
import requests

url = "https://www.alphavantage.co/query"
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": "JPM",
    "apikey": "1T2KVV7WHWGHBMQ4"
}
r = requests.get(url, params=params)
data = r.json()
print(list(data.keys()))

['Meta Data', 'Time Series (Daily)']


In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
from fredapi import Fred

# 
START = "2018-01-01"
END   = "2024-12-31"
fred  = Fred(api_key="793ea442d5187f9e5b7b0c30de96367a")

# 1. JPM
print("JPM data...")
jpm = yf.download("JPM", start=START, end=END)
jpm.columns = [c[0] for c in jpm.columns]
print(f"JPM: {len(jpm)} row")

# 2. VIX
print("VIX data...")
vix = yf.download("^VIX", start=START, end=END, auto_adjust=True)
vix.columns = [c[0] for c in vix.columns]
print(f"VIX: {len(vix)} row")

# 3. Treasury yield
print("Treasury data...")
treasury_series = {
    "DGS3MO": "Yield_3M",
    "DGS1":   "Yield_1Y",
    "DGS2":   "Yield_2Y",
    "DGS10":  "Yield_10Y"
}
treasury = pd.DataFrame()
for code, name in treasury_series.items():
    s = fred.get_series(code, observation_start=START, observation_end=END)
    treasury[name] = s
print(f"Treasury yield: {len(treasury)} row ")

print("\n Download complete")

JPM data...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


JPM: 1760 
VIX data...
VIX: 1760 
Treasury data...
Treasury yield: 1827 

 Download complete


In [9]:
# 合并数据集
# JPM只保留收盘价
jpm_close = jpm[["Close"]].copy()
jpm_close.columns = ["JPM_Close"]
jpm_close.index = pd.to_datetime(jpm_close.index)

# VIX只保留收盘价
vix_close = vix[["Close"]].copy()
vix_close.columns = ["VIX_Close"]
vix_close.index = pd.to_datetime(vix_close.index)

# 国债统一索引格式
treasury.index = pd.to_datetime(treasury.index)

# 合并，以JPM交易日为基准
master = jpm_close.copy()
master = master.join(vix_close, how="left")
master = master.join(treasury, how="left")

# 填充国债缺失值
master = master.ffill()
master = master.dropna()

print(f"主数据集：{master.shape[0]} row × {master.shape[1]} column")
print(master.tail())

主数据集：1760 row × 6 column
             JPM_Close  VIX_Close  Yield_3M  Yield_1Y  Yield_2Y  Yield_10Y
Date                                                                      
2024-12-23  231.216553  16.780001      4.36      4.26      4.30       4.59
2024-12-24  235.018570  14.270000      4.40      4.24      4.29       4.59
2024-12-26  235.823608  14.730000      4.35      4.23      4.30       4.58
2024-12-27  233.912872  15.950000      4.31      4.20      4.31       4.62
2024-12-30  232.118561  17.400000      4.37      4.17      4.24       4.55


In [10]:
# 特征工程
df = master.copy()

# 1. 日收益率
df["Log_Return"] = np.log(df["JPM_Close"] / df["JPM_Close"].shift(1))

# 2. 滚动波动率（annual）
df["RealVol_10d"] = df["Log_Return"].rolling(10).std() * np.sqrt(252)
df["RealVol_30d"] = df["Log_Return"].rolling(21).std() * np.sqrt(252)
df["RealVol_60d"] = df["Log_Return"].rolling(63).std() * np.sqrt(252)

# 3. VIX制度分类
lo = df["VIX_Close"].quantile(0.33)
hi = df["VIX_Close"].quantile(0.66)
df["VIX_Regime"] = pd.cut(df["VIX_Close"],
                           bins=[-np.inf, lo, hi, np.inf],
                           labels=["Low", "Medium", "High"])

# 4. 利率动量（10天）
df["Rate_Momentum"] = df["Yield_3M"].diff(10)

# 5. 收益率曲线斜率
df["Yield_Spread"] = df["Yield_10Y"] - df["Yield_2Y"]

# 6. VIX-JPM滚动相关性
df["VIX_JPM_Corr"] = df["VIX_Close"].rolling(30).corr(df["JPM_Close"])

df = df.dropna()

print(f"特征工程完成：{df.shape[0]} row × {df.shape[1]} column")
print(f"所有特征：{list(df.columns)}")
print(df.tail())

特征工程完成：1697 行 × 14 列
所有特征：['JPM_Close', 'VIX_Close', 'Yield_3M', 'Yield_1Y', 'Yield_2Y', 'Yield_10Y', 'Log_Return', 'RealVol_10d', 'RealVol_30d', 'RealVol_60d', 'VIX_Regime', 'Rate_Momentum', 'Yield_Spread', 'VIX_JPM_Corr']
             JPM_Close  VIX_Close  Yield_3M  Yield_1Y  Yield_2Y  Yield_10Y  \
Date                                                                         
2024-12-23  231.216553  16.780001      4.36      4.26      4.30       4.59   
2024-12-24  235.018570  14.270000      4.40      4.24      4.29       4.59   
2024-12-26  235.823608  14.730000      4.35      4.23      4.30       4.58   
2024-12-27  233.912872  15.950000      4.31      4.20      4.31       4.62   
2024-12-30  232.118561  17.400000      4.37      4.17      4.24       4.55   

            Log_Return  RealVol_10d  RealVol_30d  RealVol_60d VIX_Regime  \
Date                                                                       
2024-12-23    0.003319     0.224481     0.184694     0.300058     Medium   
2

In [11]:
df.to_csv("featured_dataset.csv")
print(" featured_dataset.csv")
print(f"dataset：{df.shape[0]} row × {df.shape[1]} column")

 featured_dataset.csv
data set：1697 row × 14 column


In [13]:
# 保存原始数据
jpm.to_csv("jpm_raw.csv")
vix.to_csv("vix_raw.csv")
treasury.to_csv("treasury_raw.csv")